# Medical Q&A Fine-Tuning with LoRA

End-to-end: dataset → preprocessing → LoRA fine-tuning → evaluation → FastAPI.

**Dataset:** `lavita/ChatDoctor-HealthCareMagic-100k` (5 000-row subset)  
**Base model:** `microsoft/BioGPT-Large`  
**PEFT:** LoRA (r=16)  
**Stack:** Transformers + PEFT + Accelerate + FastAPI + Uvicorn

## 0. Environment Setup

In [ ]:
# Install dependencies (run once)
# !pip install transformers datasets peft accelerate \
#       sacrebleu rouge-score psutil pynvml \
#       fastapi uvicorn httpx pytest pydantic \
#       pandas tqdm PyYAML

In [ ]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name != "02_medical_qa":
    for p in [Path.cwd()] + list(Path.cwd().parents):
        if (p / "src" / "preprocess.py").exists():
            os.chdir(p)
            PROJECT_ROOT = p
            break
sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("CPU only — reduce subset_size to 200 and epochs to 1 for local testing")

## 1. Configuration

In [ ]:
from src.preprocess import PreprocessConfig

preprocess_cfg = PreprocessConfig(
    dataset_name     = "lavita/ChatDoctor-HealthCareMagic-100k",
    fallback_dataset = "medalpaca/medical_meadow_medqa",
    subset_size      = 5000,   # set to 200 for fast CPU testing
    model_name       = "microsoft/BioGPT-Large",
    max_seq_length   = 512,
    train_split      = 0.90,
    val_split        = 0.05,
    test_split       = 0.05,
    seed             = 42,
    processed_dir    = "data/processed",
    cache_dir        = "data/raw",
)
print("Config OK")

## 2. Data Preprocessing

Steps:
1. Load dataset (Hub → fallback → synthetic safety net)
2. Normalise column names
3. Length filter + SHA-256 deduplication
4. Alpaca-style instruction template
5. Tokenise with label masking (`-100` on prompt tokens)
6. Train / val / test split → save to disk

In [ ]:
from src.preprocess import MedicalDataLoader, DataCleaner, InstructionFormatter

loader = MedicalDataLoader(preprocess_cfg)
raw_df = loader.load()
print(f"Loaded: {len(raw_df):,} rows")
raw_df.head(3)

In [ ]:
cleaner  = DataCleaner(preprocess_cfg)
clean_df = cleaner.clean(raw_df)
print(f"After cleaning: {len(clean_df):,} rows")

import pandas as pd
clean_df["q_len"] = clean_df["instruction"].str.split().str.len()
clean_df["a_len"] = clean_df["response"].str.split().str.len()
clean_df[["q_len","a_len"]].describe().round(1)

In [ ]:
try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    clean_df["q_len"].hist(bins=50, ax=axes[0], color="steelblue", edgecolor="white")
    axes[0].set(title="Question length (words)", xlabel="Words", ylabel="Count")
    clean_df["a_len"].hist(bins=50, ax=axes[1], color="darkorange", edgecolor="white")
    axes[1].set(title="Answer length (words)", xlabel="Words", ylabel="Count")
    plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib not installed — skipping plot")

In [ ]:
formatter    = InstructionFormatter()
formatted_ds = formatter.format(clean_df)
print("Columns:", formatted_ds.column_names)
print("\nSample formatted text:")
print(formatted_ds[0]["text"][:800])

In [ ]:
from src.preprocess import MedicalPreprocessPipeline

pipeline          = MedicalPreprocessPipeline(preprocess_cfg)
tokenised_splits  = pipeline.run()

print("\n--- Dataset shapes ---")
for split, ds in tokenised_splits.items():
    print(f"  {split:<12}: {len(ds):>5} samples | cols: {ds.column_names}")

## 3. Model & LoRA Setup

In [ ]:
from src.train import TrainConfig, build_model_and_tokenizer

train_cfg = TrainConfig(
    base_model                  = "microsoft/BioGPT-Large",
    lora_r                      = 16,
    lora_alpha                  = 32,
    lora_target_modules         = ["q_proj", "v_proj"],
    lora_dropout                = 0.05,
    processed_dir               = "data/processed",
    output_dir                  = "models/medical_qa_lora",
    num_train_epochs            = 3,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 8,
    learning_rate               = 2e-4,
    fp16                        = torch.cuda.is_available(),
    gradient_checkpointing      = True,
    seed                        = 42,
    dataloader_num_workers      = 0,
)

model, tokenizer = build_model_and_tokenizer(train_cfg)
print("Model ready.")

In [ ]:
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters : {total:,}")
print(f"Trainable (LoRA) : {trainable:,}  ({100*trainable/total:.2f}%)")

## 4. Training

> **Laptop / CPU:** reduce `subset_size=200` and `num_train_epochs=1` first.  
> **Colab T4 (5k rows, 3 epochs):** ~30–60 min.

In [ ]:
from src.train import load_datasets, build_trainer

datasets = load_datasets(train_cfg)
trainer  = build_trainer(train_cfg, model, tokenizer, datasets)

eff_batch = train_cfg.per_device_train_batch_size * train_cfg.gradient_accumulation_steps
steps_per_epoch = len(datasets["train"]) // eff_batch
print(f"Effective batch size : {eff_batch}")
print(f"Steps per epoch      : {steps_per_epoch}")
print(f"Total steps          : {steps_per_epoch * train_cfg.num_train_epochs}")

In [ ]:
# Run training (this cell takes a while — see README for cloud options)
trainer.train()
print("Training complete.")

In [ ]:
from src.train import save_final_model
save_final_model(trainer, train_cfg)
print("Saved to models/medical_qa_lora/final/")

## 5. Evaluation

Metrics: **ROUGE-1/2/L**, **BLEU**, **Exact Match**, **avg latency**

In [ ]:
from src.evaluate import MedicalQAPredictor, evaluate
from datasets import load_from_disk

predictor = MedicalQAPredictor(
    model_path     = "models/medical_qa_lora/final",
    max_new_tokens = 256,
    temperature    = 0.7,
)

formatted = load_from_disk("data/processed/formatted")
test_ds   = formatted["test"]

metrics, preds, refs = evaluate(predictor, test_ds, max_samples=100)

In [ ]:
import pandas as pd
pd.DataFrame([metrics]).T.rename(columns={0: "value"})

In [ ]:
for i in range(min(3, len(preds))):
    print(f"--- Sample {i+1} ---")
    print(f"  REF : {refs[i][:250]}")
    print(f"  PRED: {preds[i][:250]}")
    print()

## 6. Interactive Inference

In [ ]:
from src.inference import MedicalQAInference

engine = MedicalQAInference(
    model_path     = "models/medical_qa_lora/final",
    max_new_tokens = 256,
    temperature    = 0.7,
)

questions = [
    "What are the early warning signs of Type 2 diabetes?",
    "How is hypertension diagnosed and treated?",
    "What is the mechanism of action of aspirin?",
]

for q in questions:
    ans = engine.answer(q)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}")
    print("-" * 60)

## 7. FastAPI Deployment

Start the server from the project root:
```bash
uvicorn deploy.app:app --host 0.0.0.0 --port 8000 --reload
```

Custom model path via env var:
```bash
MODEL_PATH=models/medical_qa_lora/final uvicorn deploy.app:app --port 8000
```

### Endpoints
| Method | Path | Description |
|--------|------|-------------|
| GET | `/health` | Liveness probe |
| GET | `/model/info` | Model metadata |
| POST | `/answer` | Single Q&A |
| POST | `/answer/batch` | Batch Q&A (up to 16) |

In [ ]:
# Uncomment after starting the server in a separate terminal
# import httpx, json
# BASE = "http://localhost:8000"
# print(httpx.get(f"{BASE}/health").json())
# r = httpx.post(f"{BASE}/answer",
#                json={"question": "What are symptoms of hypertension?"})
# print(json.dumps(r.json(), indent=2))
print("Start server first: uvicorn deploy.app:app --port 8000")

## 8. GPU Monitoring

Run in a **separate terminal** while training:
```bash
python scripts/monitor_gpu.py --interval 5 --log logs/gpu_monitor.csv
```

In [ ]:
from pathlib import Path
import pandas as pd

logs = sorted(Path("logs").glob("gpu_monitor_*.csv"))
if logs:
    df = pd.read_csv(logs[-1])
    print(df.tail(10))
else:
    print("No GPU log found — run scripts/monitor_gpu.py during training.")

## 9. Cloud Training Quick-Reference

| Platform | GPU | VRAM | ~Time (5k/3ep) | Cost/hr |
|---|---|---|---|---|
| Google Colab (free) | T4 | 16 GB | 45 min | Free |
| Kaggle | P100 | 16 GB | 40 min | Free |
| AWS `ml.g4dn.xlarge` | T4 | 16 GB | 40 min | $0.74 |
| Azure `NC6s_v3` | V100 | 16 GB | 25 min | $0.90 |
| RunPod RTX 3090 | 3090 | 24 GB | 20 min | $0.34 |
| Laptop CPU | — | — | hours | Free |

### Colab Quick-Start
```python
!git clone <your-repo>
%cd 02_medical_qa
!pip install -q transformers datasets peft accelerate sacrebleu rouge-score
!python src/preprocess.py
!python src/train.py --config configs/training_config.yaml
```

## 10. Run Tests

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v", "--tb=short", "--no-header"],
    capture_output=True, text=True, cwd=str(PROJECT_ROOT)
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])